# PSELDNets — outdoor_siren_v8 学習（train 240 / val 80 / test 80、レビュー修正版）

**v8 = 敵対的レビュー（2026-07-14）の指摘を全て織り込んだ仕切り直し版**（v5〜v7と数値の
直接比較はしない、新しい土俵）:

1. **側×クラス独立化**: v5-v7は「siren/backup_beepは全数左・horn/bike_bellは全数右」
   という交絡があった。v8は全クラス左右50/50（train/val/testすべて）
2. **test fold新設（fold3、80本）**: valはckpt選択用、**testは卒論の最終数値を出すとき
   1回だけ**使う（valid==test問題の解消）
3. **SNR/SIR正規化を直接音W基準に固定**（反射条件でも共通=ablationを真の1要素差に）
4. **全クラスに音源ジッタ**: horn基音±5%・時間±10%、beep周波数±5%、bell f0±5%、
   車回転±20%（「クラス識別完全は同一波形の産物」批判への対処）
5. 基準物理=地面反射ON（理想剛面two-ray、対象・妨害車とも）。ラベルは直接音方向

## ⚠️ 使用前に必ず確認

1. **ランタイム → T4 GPU**
2. **Drive の `MyDrive/PSELDNets_data/` に `dataset_outdoor_siren_v8.zip` をアップロード済み**
3. セルを上から順に実行
4. **fold3（test）は学習・チューニング・ckpt選択に絶対に使わない**（最終報告の直前に
   1回だけ推論する）
5. データを変えて再学習するときは `EXP_NAME` を必ず新名に

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU未接続。ランタイムのタイプを T4 GPU に変更してください。'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Drive マウントとパス設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ==== 設定（必要ならここだけ書き換える） ====
DRIVE_DATA = '/content/drive/MyDrive/PSELDNets_data'   # zip の置き場所
DRIVE_LOGS = '/content/drive/MyDrive/PSELDNets_logs'   # 学習ログ・ckpt の永続化先
DRIVE_CKPT = '/content/drive/MyDrive/PSELDNets_ckpts'  # 事前学習ckptのキャッシュ
DATASET    = 'outdoor_siren_v8'
EXP_NAME   = 'outdoor_siren_v8_run1'                             # 固定名（resume用）
ZIP_NAME   = f'dataset_{DATASET}.zip'

import os
for d in [DRIVE_DATA, DRIVE_LOGS, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)
ZIP_PATH = f'{DRIVE_DATA}/{ZIP_NAME}'
assert os.path.exists(ZIP_PATH), f'⚠️ {ZIP_PATH} がありません。zip をアップロードしてください。'
print(f'OK: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.0f} MB)')

## 3. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既にあります: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')

## 4. パッケージインストール

`numpy` / `h5py` / `scipy` / `torch` は Colab に最初から入っています。
ここでは**触らず**、不足しているものだけ追加します（v1-v4 と同一・再起動不要）。

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

import numpy, lightning, torchmetrics, librosa
print(f'numpy {numpy.__version__} / lightning {lightning.__version__} / '
      f'torchmetrics {torchmetrics.__version__} / librosa {librosa.__version__}')

## 5. 事前学習チェックポイント（Drive キャッシュ → なければ HF から）

In [ ]:
import shutil

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'
CACHE = f'{DRIVE_CKPT}/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    if os.path.exists(CACHE):
        print('Drive キャッシュからコピー...')
        shutil.copy(CACHE, CKPT)
    else:
        print('HuggingFace からダウンロード...')
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(repo_id='Jinbo-HU/PSELDNets',
                              filename='model/mACCDOA-HTSAT-0.567.ckpt',
                              repo_type='dataset')
        shutil.copy(src, CKPT)
        shutil.copy(CKPT, CACHE)   # 次回用に Drive へキャッシュ
print(f'OK: {CKPT} ({os.path.getsize(CKPT)/1e6:.0f} MB)')

## 6. データセット展開

zip は `datasets/...` 構成なのでリポジトリ直下で解凍するだけ。
クラス辞書 `cls_indices_train.tsv`（**本データセット専用の4クラス**: Siren/Horn/
BackupBeep/BikeBell）も同梱。

In [ ]:
import zipfile

if not os.path.exists(f'datasets/{DATASET}/foa'):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('.')
    print('解凍完了')

n_foa = len(os.listdir(f'datasets/{DATASET}/foa'))
n_meta = len(os.listdir(f'datasets/{DATASET}/metadata'))
n_cls = len(open('datasets/cls_indices_train.tsv').readlines())
print(f'foa: {n_foa} / metadata: {n_meta} / classes: {n_cls}')
assert n_foa == 400 and n_meta == 400 and n_cls == 4, '⚠️ ファイル数が想定と違います'

## 7. 設定ファイル2つを作成（新規追加のみ・リポジトリ既存ファイルは無編集）

In [ ]:
data_yaml = """audio_type: foa
audio_feature: logmelIV
sample_rate: 24000
nfft: 1024
n_mels: 64
hoplen: 240
window: hann

train_chunklen_sec: 10
train_hoplen_sec: 10
test_chunklen_sec: 10
test_hoplen_sec: 10

train_dataset:
  outdoor_siren_v8: [fold1_room1]
valid_dataset:
  outdoor_siren_v8: [fold2_room1]
test_dataset:
  outdoor_siren_v8: [fold3_room1]
"""

# 開発用の推論はval(fold2)に対して行う（testを汚さないため、test=fold2の別データ設定を用意）
data_yaml_valinfer = data_yaml.replace(
    'test_dataset:\n  outdoor_siren_v8: [fold3_room1]',
    'test_dataset:\n  outdoor_siren_v8: [fold2_room1]')

exp_yaml = """# @package _global_
defaults:
 - override /data: outdoor_siren_v8.yaml
 - override /loss: multi_accdoa.yaml
 - _self_

task_name: outdoor_siren_v8

model:
  batch_size: 8
  kwargs:
    pretrained_path: ckpts/mACCDOA-HTSAT-0.567.ckpt
    audioset_pretrain: false
  optimizer:
    kwargs: {lr: 0.0003}
  lr_scheduler:
    kwargs: {step_size: 60}

trainer:
  max_epochs: 100
  check_val_every_n_epoch: 5
"""

exp_yaml_valinfer = exp_yaml.replace('override /data: outdoor_siren_v8.yaml',
                                     'override /data: outdoor_siren_v8_valinfer.yaml')

open('configs/data/outdoor_siren_v8.yaml', 'w').write(data_yaml)
open('configs/data/outdoor_siren_v8_valinfer.yaml', 'w').write(data_yaml_valinfer)
open('configs/experiment/outdoor_siren_v8.yaml', 'w').write(exp_yaml)
open('configs/experiment/outdoor_siren_v8_valinfer.yaml', 'w').write(exp_yaml_valinfer)
print('wrote configs (v8 + valinfer)')

## 8. 前処理（ラベル → HDF5、クリップ索引の作成。1分未満）

In [ ]:
IDX = f'_hdf5/data/24000fs/wav/dev/{DATASET}_10sChunklen_10sHoplen_train.csv'
if not os.path.exists(IDX):
    !python src/preproc.py dataset={DATASET}
else:
    print('既に前処理済み')
!head -3 {IDX}

## 9. 実行前チェック

In [ ]:
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt',     'チェックポイント'),
    ('datasets/cls_indices_train.tsv',      'クラス辞書 TSV (4クラス)'),
    (f'datasets/{DATASET}/foa',             'FOA データ (400)'),
    (f'datasets/{DATASET}/metadata',        'ラベル CSV (400)'),
    (f'configs/experiment/{DATASET}.yaml',  '実験設定'),
    (IDX,                                   'クリップ索引'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

## 10. 学習（T4 で 60〜90 分見込み、100epoch）

- 学習・ckpt選択に使うのは train(fold1) と val(fold2) のみ。test(fold3) は触らない
- `last.ckpt` があれば自動再開。データを変えたら `EXP_NAME` を新名に
- epoch数100はv6/v7と同じ根拠（train240本、v6はep45以降収束）

In [ ]:
LAST = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/last.ckpt'
resume = f'ckpt_path={LAST}' if os.path.exists(LAST) else ''
print('resume:', resume or '(new run)')

!python src/train.py experiment={DATASET} \
    experiment_name={EXP_NAME} \
    paths.log_dir={DRIVE_LOGS} \
    {resume}

## 11. 結果の確認（val基準）

`val/macro` の ER / F / LE / LR / SELD_scr を抽出。**この数値はv5〜v7と直接比較しない**
（側×クラス独立化・音源ジッタ・SNR基準変更で土俵が変わったため）。
v8内での比較（今後のablation条件間、対照ラン）にのみ使う。

In [ ]:
import re

log_path = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/train.log'
lines = [l for l in open(log_path, errors='ignore')
         if 'val/macro' in l or 'train: loss_all' in l]
print(f'--- {log_path} ---')
for l in lines:
    print(re.sub(r'\x1b\[[0-9;]*m', '', l).rstrip())

vals = [l for l in lines if 'val/macro' in l]
if vals:
    print('\n=== 最終 val/macro ===')
    print(re.sub(r'\x1b\[[0-9;]*m', '', vals[-1]).strip())

---
## メモ

- 生成条件の全記録はローカル `outdoor_seld_e2e/out/dataset_outdoor_siren_v8/`
  （inspection.csv には実効SNR・帯域内SIR・全長SIR・反射仰角バイアスの記録列あり）
- 分割: train=fold1(240) / val=fold2(80, ckpt選択・開発分析用) / **test=fold3(80, 最終報告
  専用・1回だけ)**
- 開発用の予測取得は下のセル12（valinfer設定でfold2に推論）を使う
- ablation条件（no-doppler等）を作る際の注意: doppler-off条件は**ラベルも一定遅延規約で
  生成**（レビューP1）。fastsimのスイッチは検証済み・P2修正済み
- **experiment_nameの使い回し禁止**（NaN崩壊事故参照）

## 12. 開発用推論（val=fold2、誤り解剖・イベント指標用）

学習後に実行。予測CSVを連結してDriveに保存 → ローカルでstep8/step8dに掛ける。
（fold3への推論は卒論の最終数値を出すときに1回だけ。そのときは
`experiment=outdoor_siren_v8 mode=test` で実行する）

In [ ]:
import glob, os
best_ckpt = sorted(glob.glob(f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/epoch_*.ckpt'))[-1]
print('using:', best_ckpt)

!python src/infer.py experiment=outdoor_siren_v8_valinfer \
    mode=test \
    ckpt_path="{best_ckpt}" \
    model.kwargs.pretrained_path=null \
    experiment_name=infer_{EXP_NAME}_val \
    paths.log_dir={DRIVE_LOGS}

# 予測CSVを1ファイルに連結してDriveへ（ローカル取得の省力化）
exp = f'infer_{EXP_NAME}_val'
sub = f'{DRIVE_LOGS}/{DATASET}/runs/{exp}/submissions'
out_lines = []
for p in sorted(glob.glob(f'{sub}/*.csv')):
    stem = os.path.basename(p)[:-4]
    for line in open(p):
        if line.strip():
            out_lines.append(f'{stem},{line.strip()}')
out = f'/content/drive/MyDrive/PSELDNets_data/{exp}_all.csv'
open(out, 'w').write('\n'.join(out_lines))
print('wrote', out, len(out_lines), 'lines')

## 13. 【任意・レビュー#8対応】対照ラン: v5データ×v6相当の学習ステップ

「v5→v6の改善はデータ量が主因」の対立仮説（総ステップ・LRスケジュール差）を潰す1本。
v5データ（train60本）のまま、v6と同じ**総ステップ3000・LR減衰1800ステップ**に揃える
（60本×batch8=8ステップ/epoch → 375epoch、step_size=225）。
前提: Driveに `dataset_outdoor_siren_v5.zip`（アップロード済みのはず）。40分前後。

In [ ]:
import zipfile, os

# v5データの展開と設定（このランタイムに無ければ作る）
if not os.path.exists('datasets/outdoor_siren_v5/foa'):
    with zipfile.ZipFile(f'{DRIVE_DATA}/dataset_outdoor_siren_v5.zip') as z:
        z.extractall('.')
    print('v5 unzipped')
    # 注意: cls_indices_train.tsv はv8と同一の4クラス辞書なので上書きでOK

v5_data = open('configs/data/outdoor_siren_v8.yaml').read().replace(
    'outdoor_siren_v8', 'outdoor_siren_v5').replace(
    '[fold3_room1]', '[fold2_room1]')   # v5にfold3は無い
open('configs/data/outdoor_siren_v5.yaml', 'w').write(v5_data)
v5_exp = open('configs/experiment/outdoor_siren_v8.yaml').read().replace(
    'outdoor_siren_v8', 'outdoor_siren_v5')
open('configs/experiment/outdoor_siren_v5.yaml', 'w').write(v5_exp)

IDX5 = '_hdf5/data/24000fs/wav/dev/outdoor_siren_v5_10sChunklen_10sHoplen_train.csv'
if not os.path.exists(IDX5):
    !python src/preproc.py dataset=outdoor_siren_v5

CTRL_EXP = 'outdoor_siren_v5_run3_stepmatched'
LAST5 = f'{DRIVE_LOGS}/outdoor_siren_v5/runs/{CTRL_EXP}/checkpoints/last.ckpt'
resume5 = f'ckpt_path={LAST5}' if os.path.exists(LAST5) else ''

!python src/train.py experiment=outdoor_siren_v5 \
    experiment_name={CTRL_EXP} \
    trainer.max_epochs=375 \
    model.lr_scheduler.kwargs.step_size=225 \
    paths.log_dir={DRIVE_LOGS} \
    {resume5}